**This is a tutorial on cleaning data for marketing analysis**

In [5]:
import numpy as np
import pandas as pd

First let's get the data

In [22]:
df = pd.read_csv('/content/marketing_campaign_data_messy.csv')
print('number of rows ', df.shape[0],' and number of columns ',df.shape[1])

number of rows  2020  and number of columns  12


In [23]:
df.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


**Step 1 Header cleaning**

In [24]:
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']


In [25]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
print('Fix applied')
print(df.columns.tolist())

Fix applied
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


**Step 2: Type conversion & currency cleaning**

In [26]:
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22


In [27]:
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]','',regex=True)
df['spend'] = pd.to_numeric(df['spend'])
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


**Step 3: Categorical typos (FUZZY LOGIC)**

In [28]:
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']


In [29]:
cleanup_map = {
    'Tik_Tok' : 'TikTok',
    'Facebok' : 'Facebook',
    'E-mail' : 'Email',
    'Insta_gram' : 'Instagram',
    'Gogle' : 'Google Ads',
    'N\A' : np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)
print("FIX APPLIED")
print(df['channel'].unique())

FIX APPLIED
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


<>:7: SyntaxWarning: invalid escape sequence '\A'
<>:7: SyntaxWarning: invalid escape sequence '\A'
/tmp/ipython-input-2298516275.py:7: SyntaxWarning: invalid escape sequence '\A'
  'N\A' : np.nan


**Handling mixed booleans**

In [30]:
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']


In [31]:
bool_map= {
    'Y' : True,
    '0' : False,
    'No' : False,
    'Yes' : True,
    '1' : True,
    'False' : False,
    'True' : True
}
df['active'] = df['active'].replace(bool_map)
print('FIXES Aplied')
print(df['active'].unique())

FIXES Aplied
[ True False]


/tmp/ipython-input-3660402396.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].replace(bool_map)


**Date parsing**

In [32]:
print(df['start_date'].dtypes)
print(df['end_date'].dtypes)

object
object


In [33]:
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("FIX APPLIED")
print(df['start_date'].dtypes)
print(df['end_date'].dtypes)

FIX APPLIED
datetime64[ns]
datetime64[ns]


/tmp/ipython-input-734643689.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


**STEP 6 : Logical integrity (clicks vs impressions)**

In [34]:
  df = df.loc[:, ~df.columns.duplicated()]

In [37]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


In [38]:
impossible_mask = df['clicks']>df['impressions']
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


**STEP 7 : Logical integrity (time travel)**

In [39]:
time_travel_mask = df['end_date']<df['start_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27


In [40]:
df.loc[time_travel_mask, 'end_date']= df.loc[time_travel_mask, 'start_date']+pd.Timedelta(days=30)
print('FIX APPLIED')
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

FIX APPLIED
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


**STEP : 8
Handeling outlirs(winsorizing)**

In [45]:
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)
IQR = Q3 - Q1 #usual range
upper_limit = Q3 +(3 * IQR)
outlier_mask = df['spend']>upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

8603.537499999999
     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
FIX APPLIED


In [46]:
print('FIX APPLIED')
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

FIX APPLIED
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


**STEP : 9
String parsing (feature extaction)**

In [47]:
print(df['campaign_name'].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object


In [49]:
df['season']= df['campaign_name'].str.extract(r'Q\d_([^_]+)_')
print('FIX APPLIED')
print(df[['campaign_name','season']])

FIX APPLIED
                 campaign_name       season
0          Q4_Summer_CMP-00001       Summer
1          Q1_Launch_CMP-00002       Launch
2          Q3_Winter_CMP-00003       Winter
3     Q1_BlackFriday_CMP-00004  BlackFriday
4          Q2_Winter_CMP-00005       Winter
...                        ...          ...
2015       Q3_Summer_CMP-00400       Summer
2016       Q4_Summer_CMP-01255       Summer
2017       Q2_Launch_CMP-01050       Launch
2018       Q4_Winter_CMP-01118       Winter
2019       Q4_Launch_CMP-01554       Launch

[2020 rows x 2 columns]
